<a href="https://colab.research.google.com/github/keshariujjwal51/Ujjwal/blob/main/Assignment_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Libraries

In [1]:
!pip install -q langchain langchain-community langchain-text-splitters pypdf sentence-transformers faiss-cpu transformers accelerate


In [2]:
!pip install --upgrade transformers

## Import Libraries

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import torch
from transformers import pipeline

print("All imports successful!")


/tmp/ipykernel_3445/1322584803.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


All imports successful!


After running the previous cell, the Colab runtime will restart. Please run the cells from the top to continue.

In [4]:
from google.colab import files

uploaded = files.upload()

Saving Res.pdf to Res (1).pdf


In [5]:
import os

# Grab the filename dynamically instead of hardcoding it,
# so this works regardless of what file name you upload.
pdf_filename = list(uploaded.keys())[0]
print("Uploaded file:", pdf_filename)


Uploaded file: Res (1).pdf


In [6]:
loader = PyPDFLoader(pdf_filename)

documents = loader.load()

print(documents[0].page_content)


UJJWAL KESHARI 
+91 9122440083 | keshariujjwal51@gmail.com 
LinkedIn: linkedin.com/in/ujjwal-keshari-6b522723b 
Greater Noida | Utter Pradesh  
 
PROFESSIONAL SUMMARY 
Results-driven PGDM Marketing candidate with strong expertise in consumer behaviour analysis, market research, and 
digital engagement. Proven ability to analyse audience trends, generate data-driven insights. Adept at translating 
analytical findings into actionable marketing strategies. 
 
SKILLS 
Technical Skills: Advanced Excel, Data Analysis, Digital Marketing 
Core Competencies: Consumer Behaviour Analysis, Market Research, Customer Segmentation 
Soft Skills: Leadership, Communication, Team Collaboration, Critical Thinking, Time Management 
 
INTERNSHIP EXPERIENCE 
Trainee – 1M1B AI & Sustainability Internship 
Dec 2025 – Jan 2026 
 Developed an AI-based solution for sustainable agriculture, enabling rainfall prediction and crop disease 
detection 
 Analysed agricultural and environmental datasets to generate act

## Split the Text into Chunks

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100  # matches the value documented in the report
)

chunks = text_splitter.split_documents(
    documents
)

print(
    "Number of Chunks:",
    len(chunks)
)


Number of Chunks: 6


## Convert Chunks into Text

In [8]:
texts = [
    chunk.page_content
    for chunk in chunks
]

print(texts[0])

UJJWAL KESHARI 
+91 9122440083 | keshariujjwal51@gmail.com 
LinkedIn: linkedin.com/in/ujjwal-keshari-6b522723b 
Greater Noida | Utter Pradesh  
 
PROFESSIONAL SUMMARY 
Results-driven PGDM Marketing candidate with strong expertise in consumer behaviour analysis, market research, and 
digital engagement. Proven ability to analyse audience trends, generate data-driven insights. Adept at translating 
analytical findings into actionable marketing strategies. 
 
SKILLS


## Create Embeddings

In [9]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = model.encode(
    texts,
    convert_to_numpy=True
)

print(
    embeddings.shape
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(6, 384)


## Create FAISS Vector Database

In [10]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    np.array(embeddings).astype("float32")
)

print(
    "Embeddings Stored Successfully!"
)


Embeddings Stored Successfully!


## Ask a Question

In [11]:
query = """What are the skills mentioned in the document?""".strip()

print(query)


What are the skills mentioned in the document?


## Convert Question to Embedding

In [12]:
query_embedding = model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")


## Retrieve Top 3 Chunks

In [13]:
k = 3

distances, indices = index.search(
    query_embedding,
    k
)

print("Indices:", indices)
print("Distances:", distances)


Indices: [[1 5 4]]
Distances: [[1.3253981 1.4994979 1.6691782]]


## Display Retrieved Chunks

In [14]:
retrieved_chunks = []

for idx in indices[0]:
    retrieved_chunks.append(
        texts[idx]
    )

for chunk in retrieved_chunks:
    print(chunk)
    print("-"*50)

analytical findings into actionable marketing strategies. 
 
SKILLS 
Technical Skills: Advanced Excel, Data Analysis, Digital Marketing 
Core Competencies: Consumer Behaviour Analysis, Market Research, Customer Segmentation 
Soft Skills: Leadership, Communication, Team Collaboration, Critical Thinking, Time Management 
 
INTERNSHIP EXPERIENCE 
Trainee – 1M1B AI & Sustainability Internship 
Dec 2025 – Jan 2026
--------------------------------------------------
Lloyd Business School | 2025 – Present 
 
Bachelor’s Degree | BBA  
School of Management Science, Varanasi | 2021 – 2024 
 
CERTIFICATIONS 
 IBM Cloud Certification 
 Google Digital Marketing Certification 
 LinkedIn Learning – AI Agents 
 ADCA (Advanced Diploma in Computer Applications) 
 Fundamentals of Digital Marketing – Google Digital Garage
--------------------------------------------------
 Identified key decision drivers including cost, infrastructure, and environmental concerns 
Sustainable Development Project 
 De

## Load the Language Model


In [16]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# FLAN-T5 is a sequence-to-sequence (encoder-decoder) model, so it needs the
# "text2text-generation" pipeline, NOT "text-generation" (which is for
# decoder-only causal LMs like GPT). Using the wrong pipeline type here was
# the reason the previous version echoed the prompt instead of answering.
device = 0 if torch.cuda.is_available() else -1  # use the T4 GPU if available

# The 'text2text-generation' task is not being recognized by the pipeline.
# This often indicates an environment issue or a specific library version problem.
# As a workaround, we will load the model and tokenizer directly and create a custom generator.

model_name = "google/flan-t5-base"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

if device != -1: # if GPU is available
    model.to(f'cuda:{device}')

# Define a custom generator function that mimics the pipeline's behavior
def custom_generator(prompts, max_new_tokens):
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    outputs = model.generate(
        inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens
    )
    decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return [{"generated_text": text} for text in decoded_outputs]

# Assign the custom generator
generator = custom_generator

print("FLAN-T5 model loaded and custom generator initialized.")


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 model loaded and custom generator initialized.


## Prompt

In [17]:
context = "\n\n".join(retrieved_chunks)

prompt = f"""Answer the question using ONLY the context below.
If the answer is not contained in the context, say "I don't know".

Context:
{context}

Question:
{query}

Answer:"""


## Final Answer

In [18]:
answer = generator(
    prompt,
    max_new_tokens=150
)

print(answer[0]["generated_text"].strip())


Advanced Excel, Data Analysis, Digital Marketing


## Conclusion

This project successfully developed and implemented a Retrieval-Augmented Generation (RAG) system that combines document retrieval and natural language generation to provide accurate and context-aware answers from custom PDF documents. The pipeline was built using LangChain for workflow management, Sentence Transformers for generating semantic embeddings, FAISS for efficient similarity-based document retrieval, and FLAN-T5 as the language generation model.

The system effectively extracts relevant information from the uploaded documents, retrieves the most meaningful content based on user queries, and uses the language model to generate precise responses. This approach reduces the chances of generating irrelevant or incorrect answers by grounding the responses in the provided document knowledge.

Overall, the project demonstrates the effectiveness of RAG architecture in building intelligent question-answering systems. It can be further enhanced by integrating larger language models, improving document preprocessing techniques, supporting multiple document formats, and optimizing retrieval methods for better accuracy and scalability. This implementation provides a strong foundation for developing real-world AI-powered knowledge retrieval applications.
